# Hypothesis Test — Augmentation Strength × Regularizer Strength

**Hypothesis.** *"Weak MixStyle pairs best with strong regularization;
strong MixStyle pairs best with weak regularization."*

This notebook runs a self-contained 2×4 grid:

|                              | Logit KL | Multiscale | MMD-only | KL + MMD |
| ---------------------------- | :------: | :--------: | :------: | :------: |
| **Strong MS** (p=0.75, a=0.1) | sweep    | sweep      | sweep    | sweep    |
| **Weak MS**  (p=0.5, a=0.5)   | sweep    | sweep      | sweep    | sweep    |

All checkpoints and result JSONs are written to a **dedicated
`artifacts/hypothesis_test/` tree** so the existing
`artifacts/results/` (which is stale relative to the partially-deleted
`brats23_npz_dg/`) is untouched. The data integrity cell will detect
missing `.npz` files and re-run preprocessing if needed.

**Total runs:** 57 (3 references + 16 KL + 10 multiscale + 10 MMD +
18 combined). At ~30 min/run that's ~28 h — plan for two
overnights, or drop the combined grid to 2×2 to fit one.


## Setup

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "synapseclient": "synapseclient",
    "nibabel": "nibabel",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "torchvision": "torchvision",
}
missing = [pkg for module_name, pkg in required.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    raise RuntimeError(
        "Installed missing packages. Restart the kernel, then rerun the notebook from the top."
    )
print("All required packages already available.")


All required packages already available.


In [2]:
from pathlib import Path
import json
import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dg import config, runtime
from dg.splits import build_main_split, summarize_split
from dg.experiment import (
    DataSplitConfig,
    ModelConfig,
    prepare_dataloaders_from_config,
    report_experiments,
    run_experiment,
)
from dg.training import set_seed


## Output isolation — monkey-patch `config`

`dg/experiment.py` reads `config.CHECKPOINT_DIR` and
`config.RESULTS_DIR` *at call time*, so a single redirect at the top
of the notebook routes every saved result and checkpoint into a
dedicated subtree. The existing stale `artifacts/results/` and
`artifacts/checkpoints/` are not touched.

In [3]:
HT_ROOT = Path("artifacts/hypothesis_test")
HT_RESULTS = HT_ROOT / "results"
HT_CKPTS   = HT_ROOT / "checkpoints"
HT_FIGURES = HT_ROOT / "figures"
for p in (HT_RESULTS, HT_CKPTS, HT_FIGURES):
    p.mkdir(parents=True, exist_ok=True)

config.RESULTS_DIR    = str(HT_RESULTS)
config.CHECKPOINT_DIR = str(HT_CKPTS)

set_seed(config.EXP_SEED)
print("device       =", runtime.DEVICE)
print("results dir  =", config.RESULTS_DIR)
print("ckpt dir     =", config.CHECKPOINT_DIR)


device       = cuda
results dir  = artifacts\hypothesis_test\results
ckpt dir     = artifacts\hypothesis_test\checkpoints


## Data integrity / regeneration

Same pattern as `final_report_multiscale.ipynb`: read the index, check
each `.npz` exists, and re-run `preprocess_zip_to_npz()` if any are
missing. Requires `config.ZIP_PATH` to point at the BraTS ZIP.

In [4]:
from dg.preprocessing import preprocess_zip_to_npz

index_path = Path(config.OUT_ROOT) / "index.csv"

def _scan(df):
    exists = df["npz_path"].apply(lambda p: Path(p).exists())
    return int(exists.sum()), int((~exists).sum())

def _load_and_check():
    if not index_path.exists():
        return None, None, None
    df = pd.read_csv(index_path)
    present, missing = _scan(df)
    return df, present, missing

index_df, present, missing = _load_and_check()
if index_df is None or missing > 0:
    if index_df is None:
        print("No index.csv — running preprocess_zip_to_npz() from the ZIP.")
    else:
        print(f"Stale dataset: {present}/{present+missing} files present; {missing} missing.")
        print("Re-running preprocess_zip_to_npz() to regenerate from the ZIP.")
    index_df = preprocess_zip_to_npz()
    index_df, present, missing = _load_and_check()
    assert index_df is not None and missing == 0, (
        f"Preprocessing did not close the gap — {missing} rows still missing. "
        "Check config.ZIP_PATH and disk space, then re-run this cell."
    )

print("sites:", sorted(index_df["site"].unique().tolist()))
print("cases:", index_df["case_id"].nunique(), "slices:", len(index_df))
print("files on disk:", present)


sites: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
cases: 1251 slices: 1251
files on disk: 1251


## Build the main split (site 18 held out)

In [5]:
TEST_SITES = [18]
split = build_main_split(index_df, test_sites=TEST_SITES)

summarize_split("train", split["train_df"])
summarize_split("val", split["val_df"])
summarize_split("in-domain test", split["in_domain_test_df"])
summarize_split("out-domain test", split["out_domain_test_df"])

default_data_config = DataSplitConfig(
    name="default",
    train_cases=frozenset(split["train_set"]),
    validation_cases=frozenset(split["val_set"]),
    test_cases=frozenset(split["in_domain_test_set"]),
    out_domain_test_cases=frozenset(split["out_domain_test_set"]),
)



train
cases=296, slices=23559, tumor=11737, normal=11822, tumor_ratio=0.4982
top sites by cases:
site
1     44
4     33
21    25
13    25
6     24
20    23
16    22
5     16
3     11
11    10
Name: case_id, dtype: int64

val
cases=63, slices=5011, tumor=2491, normal=2520, tumor_ratio=0.4971
top sites by cases:
site
1     10
4      7
21     5
6      5
20     5
13     5
16     4
5      3
11     2
15     2
Name: case_id, dtype: int64

in-domain test
cases=63, slices=5018, tumor=2517, normal=2501, tumor_ratio=0.5016
top sites by cases:
site
1     10
4      7
21     5
6      5
20     5
13     5
16     4
5      3
11     2
15     2
Name: case_id, dtype: int64

out-domain test
cases=382, slices=30232, tumor=14980, normal=15252, tumor_ratio=0.4955
top sites by cases:
site
18    382
Name: case_id, dtype: int64


## Helper builders

Four pure functions for the four regularizer families. Each returns a
`ModelConfig` whose name encodes the (p, a, hyperparam) tuple so the
saved JSONs are easy to glob and tabulate later.

In [6]:
def _stem(p, a):
    return f"p{p}-a{a}"

def kl_cfg(p, a, lam):
    return ModelConfig(
        name=f"HT-KL-{_stem(p,a)}-lam{lam}",
        train_layers=(2,),
        use_mixstyle=True,
        mixstyle_p=p,
        mixstyle_a=a,
        consistency_lambda=lam,
    )

def multi_cfg(p, a, w):
    return ModelConfig(
        name=f"HT-Multi-{_stem(p,a)}-w{w}",
        train_layers=(2,),
        use_mixstyle=True,
        mixstyle_p=p,
        mixstyle_a=a,
        multiscale_layer_weights=(
            ("layer2", w),
            ("layer3", w),
            ("layer4", w),
            ("feats",  w),
        ),
        multiscale_distance="cosine",
        multiscale_logit_kl_lambda=0.0,
    )

def mmd_cfg(p, a, lam_mmd):
    return ModelConfig(
        name=f"HT-MMD-{_stem(p,a)}-lam{lam_mmd}",
        train_layers=(2,),
        use_mixstyle=True,
        mixstyle_p=p,
        mixstyle_a=a,
        mmd_lambda=lam_mmd,
        mmd_class_conditional=True,
    )

def consist_mmd_cfg(p, a, lam_c, lam_mmd):
    return ModelConfig(
        name=f"HT-KLMMD-{_stem(p,a)}-c{lam_c}-m{lam_mmd}",
        train_layers=(2,),
        use_mixstyle=True,
        mixstyle_p=p,
        mixstyle_a=a,
        consistency_lambda=lam_c,
        mmd_lambda=lam_mmd,
        mmd_class_conditional=True,
    )

def baseline_cfg():
    return ModelConfig(name="HT-Baseline-L2", train_layers=(2,), use_mixstyle=False)

def plain_ms_cfg(p, a):
    return ModelConfig(
        name=f"HT-MixStyle-{_stem(p,a)}",
        train_layers=(2,),
        use_mixstyle=True,
        mixstyle_p=p,
        mixstyle_a=a,
    )

# The two MixStyle regimes under test
STRONG_MS = (0.75, 0.1)
WEAK_MS   = (0.5,  0.5)


## Run-and-skip helper

Skips re-training if a result JSON already exists in the dedicated
results directory — makes the notebook safely re-runnable.

In [7]:
def run_or_skip(cfg, epochs=15):
    out_path = HT_RESULTS / f"{cfg.name}.json"
    if out_path.exists():
        print(f"[skip] {cfg.name} (cached at {out_path})")
        return json.loads(out_path.read_text(encoding="utf-8"))
    return run_experiment(cfg, default_data_config, index_df=index_df, epochs=epochs)


## Reference runs (3)

Baseline + plain MixStyle at both MS configs.

In [8]:
reference_cfgs = [
    baseline_cfg(),
    plain_ms_cfg(*STRONG_MS),
    plain_ms_cfg(*WEAK_MS),
]
ref_results = [run_or_skip(c) for c in reference_cfgs]



Data Config: default
  train cases=296 slices=23559 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  val cases=63 slices=5011 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  test cases=63 slices=5018 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  out_domain_test cases=382 slices=30232 sites=[18]


c:\Users\34783\.conda\envs\18662\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\34783\.conda\envs\18662\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Model Config: HT-Baseline-L2
  train_layers=[2]
  use_mixstyle=False mixstyle_p=0.5 mixstyle_a=0.1
  insert_after=()
  trainable_params=526594
  trainable_modules=['backbone.layer2.0.conv1.weight', 'backbone.layer2.0.bn1.weight', 'backbone.layer2.0.bn1.bias', 'backbone.layer2.0.conv2.weight', 'backbone.layer2.0.bn2.weight', 'backbone.layer2.0.bn2.bias', 'backbone.layer2.0.downsample.0.weight', 'backbone.layer2.0.downsample.1.weight', 'backbone.layer2.0.downsample.1.bias', 'backbone.layer2.1.conv1.weight', 'backbone.layer2.1.bn1.weight', 'backbone.layer2.1.bn1.bias'] ...


100%|██████████| 185/185 [00:11<00:00, 16.36it/s]


[HT-Baseline-L2] Epoch 1/15 | train_loss=0.2922 train_acc=0.8784 val_acc=0.8821


...

[HT-MixStyle-p0.5-a0.5] Epoch 9/15 | train_loss=0.0372 train_acc=0.9903 val_acc=0.9062


100%|██████████| 185/185 [00:10<00:00, 17.80it/s]


[HT-MixStyle-p0.5-a0.5] Epoch 10/15 | train_loss=0.0273 train_acc=0.9937 val_acc=0.9068


100%|██████████| 185/185 [00:10<00:00, 17.67it/s]


[HT-MixStyle-p0.5-a0.5] Epoch 11/15 | train_loss=0.0257 train_acc=0.9941 val_acc=0.9016
[HT-MixStyle-p0.5-a0.5] Early stopping at epoch 11 (patience=3)


## Logit-KL sweep (14 runs)
λ ∈ {0.05, 0.1, 0.5, 1.0, 2.0, 3.0, 5.0} at both MS configs.

In [9]:
KL_LAMBDAS = [0.05, 0.1, 0.5, 1.0, 2.0, 3.0, 5.0]
kl_cfgs = [kl_cfg(*ms, lam) for ms in (STRONG_MS, WEAK_MS) for lam in KL_LAMBDAS]
print(f"running {len(kl_cfgs)} KL configs")
kl_results = [run_or_skip(c) for c in kl_cfgs]


running 14 KL configs

Data Config: default
  train cases=296 slices=23559 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  val cases=63 slices=5011 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  test cases=63 slices=5018 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  out_domain_test cases=382 slices=30232 sites=[18]

Model Config: HT-KL-p0.75-a0.1-lam0.05
  train_layers=[2]
  use_mixstyle=True mixstyle_p=0.75 mixstyle_a=0.1
  insert_after=('layer2',)
  trainable_params=526594
  trainable_modules=['backbone.layer2.0.conv1.weight', 'backbone.layer2.0.bn1.weight', 'backbone.layer2.0.bn1.bias', 'backbone.layer2.0.conv2.weight', 'backbone.layer2.0.bn2.weight', 'backbone.layer2.0.bn2.bias', 'backbone.layer2.0.downsample.0.weight', 'backbone.layer2.0.downsample.1.weight', 'backbone.layer2.0.downsample.1.bias', 'backbone.layer2.1.conv1.weight', 'backbone.layer2.1.bn1.weight

100%|██████████| 185/185 [00:13<00:00, 13.86it/s]


[HT-KL-p0.75-a0.1-lam0.05] Epoch 1/15 | train_loss=0.3195 ce=0.3189 kl=0.0119 train_acc=0.8637 val_acc=0.8825


100%|██████████| 185/185 [00:13<00:00, 13.92it/s]


[HT-KL-p0.75-a0.1-lam0.05] Epoch 2/15 | train_loss=0.2195 ce=0.2190 kl=0.0091 train_acc=0.9122 val_acc=0.9028


...

[HT-KL-p0.5-a0.5-lam5.0] Epoch 5/15 | train_loss=0.1773 ce=0.1492 kl=0.0056 train_acc=0.9455 val_acc=0.9026


100%|██████████| 185/185 [00:13<00:00, 14.06it/s]


[HT-KL-p0.5-a0.5-lam5.0] Epoch 6/15 | train_loss=0.1616 ce=0.1348 kl=0.0054 train_acc=0.9499 val_acc=0.9008


100%|██████████| 185/185 [00:13<00:00, 14.09it/s]


[HT-KL-p0.5-a0.5-lam5.0] Epoch 7/15 | train_loss=0.1508 ce=0.1218 kl=0.0058 train_acc=0.9575 val_acc=0.9016
[HT-KL-p0.5-a0.5-lam5.0] Early stopping at epoch 7 (patience=3)


## Multiscale weight sweep (10 runs)
Uniform per-stage `w` ∈ {0.25, 0.5, 1.0, 2.0, 4.0} at both MS configs.

In [10]:
MS_WEIGHTS = [0.25, 0.5, 1.0, 2.0, 4.0]
multi_cfgs = [multi_cfg(*ms, w) for ms in (STRONG_MS, WEAK_MS) for w in MS_WEIGHTS]
print(f"running {len(multi_cfgs)} multiscale configs")
multi_results = [run_or_skip(c) for c in multi_cfgs]


running 10 multiscale configs

Data Config: default
  train cases=296 slices=23559 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  val cases=63 slices=5011 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  test cases=63 slices=5018 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  out_domain_test cases=382 slices=30232 sites=[18]

Model Config: HT-Multi-p0.75-a0.1-w0.25
  train_layers=[2]
  use_mixstyle=True mixstyle_p=0.75 mixstyle_a=0.1
  insert_after=('layer2',)
  trainable_params=526594
  trainable_modules=['backbone.layer2.0.conv1.weight', 'backbone.layer2.0.bn1.weight', 'backbone.layer2.0.bn1.bias', 'backbone.layer2.0.conv2.weight', 'backbone.layer2.0.bn2.weight', 'backbone.layer2.0.bn2.bias', 'backbone.layer2.0.downsample.0.weight', 'backbone.layer2.0.downsample.1.weight', 'backbone.layer2.0.downsample.1.bias', 'backbone.layer2.1.conv1.weight', 'backbone.layer2.1.b

100%|██████████| 185/185 [00:13<00:00, 13.31it/s]


[HT-Multi-p0.75-a0.1-w0.25] Epoch 1/15 | train_loss=0.3736 ce=0.3264 feat=0.0472 kl=0.0000 train_acc=0.8606 val_acc=0.8759


100%|██████████| 185/185 [00:13<00:00, 13.65it/s]


[HT-Multi-p0.75-a0.1-w0.25] Epoch 2/15 | train_loss=0.3039 ce=0.2374 feat=0.0665 kl=0.0000 train_acc=0.9056 val_acc=0.8978


...

[HT-Multi-p0.5-a0.5-w4.0] Epoch 13/15 | train_loss=1.9572 ce=0.2149 feat=1.7423 kl=0.0000 train_acc=0.9197 val_acc=0.9012


100%|██████████| 185/185 [00:13<00:00, 13.83it/s]


[HT-Multi-p0.5-a0.5-w4.0] Epoch 14/15 | train_loss=1.6761 ce=0.2036 feat=1.4725 kl=0.0000 train_acc=0.9252 val_acc=0.8976


100%|██████████| 185/185 [00:13<00:00, 13.65it/s]


[HT-Multi-p0.5-a0.5-w4.0] Epoch 15/15 | train_loss=2.0604 ce=0.2024 feat=1.8580 kl=0.0000 train_acc=0.9242 val_acc=0.8948


## MMD-only sweep (10 runs)
λ_mmd ∈ {0.025, 0.05, 0.1, 0.25, 0.5} at both MS configs.

In [11]:
MMD_LAMBDAS = [0.025, 0.05, 0.1, 0.25, 0.5]
mmd_cfgs = [mmd_cfg(*ms, lam) for ms in (STRONG_MS, WEAK_MS) for lam in MMD_LAMBDAS]
print(f"running {len(mmd_cfgs)} MMD configs")
mmd_results = [run_or_skip(c) for c in mmd_cfgs]


running 10 MMD configs

Data Config: default
  train cases=296 slices=23559 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  val cases=63 slices=5011 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  test cases=63 slices=5018 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  out_domain_test cases=382 slices=30232 sites=[18]

Model Config: HT-MMD-p0.75-a0.1-lam0.025
  train_layers=[2]
  use_mixstyle=True mixstyle_p=0.75 mixstyle_a=0.1
  insert_after=('layer2',)
  trainable_params=526594
  trainable_modules=['backbone.layer2.0.conv1.weight', 'backbone.layer2.0.bn1.weight', 'backbone.layer2.0.bn1.bias', 'backbone.layer2.0.conv2.weight', 'backbone.layer2.0.bn2.weight', 'backbone.layer2.0.bn2.bias', 'backbone.layer2.0.downsample.0.weight', 'backbone.layer2.0.downsample.1.weight', 'backbone.layer2.0.downsample.1.bias', 'backbone.layer2.1.conv1.weight', 'backbone.layer2.1.bn1.wei

100%|██████████| 185/185 [01:38<00:00,  1.88it/s]


[HT-MMD-p0.75-a0.1-lam0.025] Epoch 1/15 | train_loss=0.3598 ce=0.3191 mmd=1.6288 train_acc=0.8632 val_acc=0.8797


100%|██████████| 185/185 [01:39<00:00,  1.85it/s]


[HT-MMD-p0.75-a0.1-lam0.025] Epoch 2/15 | train_loss=0.2595 ce=0.2187 mmd=1.6313 train_acc=0.9127 val_acc=0.9024


...

[HT-MMD-p0.5-a0.5-lam0.5] Epoch 4/15 | train_loss=0.7452 ce=0.1678 mmd=1.1547 train_acc=0.9420 val_acc=0.9056


100%|██████████| 185/185 [01:39<00:00,  1.85it/s]


[HT-MMD-p0.5-a0.5-lam0.5] Epoch 5/15 | train_loss=0.6684 ce=0.1515 mmd=1.0339 train_acc=0.9512 val_acc=0.9066


100%|██████████| 185/185 [01:39<00:00,  1.86it/s]


[HT-MMD-p0.5-a0.5-lam0.5] Epoch 6/15 | train_loss=0.6039 ce=0.1353 mmd=0.9371 train_acc=0.9595 val_acc=0.8886
[HT-MMD-p0.5-a0.5-lam0.5] Early stopping at epoch 6 (patience=3)


## KL + MMD combined grid (18 runs)
3×3 over (λ_c, λ_mmd) at both MS configs. Routes to the
ported `train_epoch_consistency_mmd` loop.

In [12]:
KLMMD_LAMC   = [0.5, 1.0, 2.0]
KLMMD_LAMMMD = [0.025, 0.05, 0.1]
combined_cfgs = [
    consist_mmd_cfg(*ms, lc, lm)
    for ms in (STRONG_MS, WEAK_MS)
    for lc in KLMMD_LAMC
    for lm in KLMMD_LAMMMD
]
print(f"running {len(combined_cfgs)} KL+MMD configs")
combined_results = [run_or_skip(c) for c in combined_cfgs]


running 18 KL+MMD configs

Data Config: default
  train cases=296 slices=23559 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  val cases=63 slices=5011 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  test cases=63 slices=5018 sites=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23]
  out_domain_test cases=382 slices=30232 sites=[18]

Model Config: HT-KLMMD-p0.75-a0.1-c0.5-m0.025
  train_layers=[2]
  use_mixstyle=True mixstyle_p=0.75 mixstyle_a=0.1
  insert_after=('layer2',)
  trainable_params=526594
  trainable_modules=['backbone.layer2.0.conv1.weight', 'backbone.layer2.0.bn1.weight', 'backbone.layer2.0.bn1.bias', 'backbone.layer2.0.conv2.weight', 'backbone.layer2.0.bn2.weight', 'backbone.layer2.0.bn2.bias', 'backbone.layer2.0.downsample.0.weight', 'backbone.layer2.0.downsample.1.weight', 'backbone.layer2.0.downsample.1.bias', 'backbone.layer2.1.conv1.weight', 'backbone.layer2.1

100%|██████████| 185/185 [01:37<00:00,  1.90it/s]


[HT-KLMMD-p0.75-a0.1-c0.5-m0.025] Epoch 1/15 | train_loss=0.3649 ce=0.3189 kl=0.0106 mmd=1.6303 train_acc=0.8634 val_acc=0.8791


100%|██████████| 185/185 [01:39<00:00,  1.87it/s]


[HT-KLMMD-p0.75-a0.1-c0.5-m0.025] Epoch 2/15 | train_loss=0.2637 ce=0.2187 kl=0.0083 mmd=1.6322 train_acc=0.9136 val_acc=0.9010


...

[HT-KLMMD-p0.5-a0.5-c2.0-m0.1] Epoch 4/15 | train_loss=0.3097 ce=0.1415 kl=0.0043 mmd=1.5967 train_acc=0.9487 val_acc=0.9096


100%|██████████| 185/185 [01:38<00:00,  1.88it/s]


[HT-KLMMD-p0.5-a0.5-c2.0-m0.1] Epoch 5/15 | train_loss=0.2846 ce=0.1184 kl=0.0041 mmd=1.5794 train_acc=0.9583 val_acc=0.9074


100%|██████████| 185/185 [01:45<00:00,  1.76it/s]


[HT-KLMMD-p0.5-a0.5-c2.0-m0.1] Epoch 6/15 | train_loss=0.2697 ce=0.1000 kl=0.0051 mmd=1.5954 train_acc=0.9644 val_acc=0.9040
[HT-KLMMD-p0.5-a0.5-c2.0-m0.1] Early stopping at epoch 6 (patience=3)
